# Notebook 1: Business Understanding & Data Audit

# Hotel Revenue Management & Dynamic Pricing Intelligence

---

## Project Workflow

- ✅ Notebook 1 — Business Understanding & Data Audit *(Current Notebook)*
- Notebook 2 — Data Cleaning & Feature Engineering
- Notebook 3 — Exploratory Data Analysis
- Notebook 4 — SQL Business Analysis
- Notebook 5 — Revenue Management Simulation

---

## Objective

The objective of this notebook is to understand the business problem, examine the dataset structure, assess data quality, and identify the analytical requirements for building a hotel revenue management decision support system.

---

## Business Understanding

### Project Overview

Hotels often rely on fixed room pricing despite significant fluctuations in customer demand throughout the year. Factors such as booking lead time, occupancy levels, seasonality, cancellations, and customer segments directly influence revenue potential. Without adaptive pricing strategies, hotels risk losing revenue during high-demand periods and leave rooms unsold during periods of low occupancy.

This project develops a **Revenue Management Decision Support System** that analyzes historical booking patterns, reconstructs occupancy using reservation-level data, and evaluates rule-based dynamic pricing strategies. The final solution enables hotel executives to simulate pricing scenarios and estimate their impact on occupancy and revenue.

### Business Objectives

- Understand historical booking behaviour and demand patterns.
- Analyze occupancy, cancellations, and revenue drivers.
- Identify pricing opportunities using lead time and occupancy.
- Simulate rule-based dynamic pricing strategies.
- Compare static and dynamic pricing approaches.
- Deliver interactive executive dashboards for decision support.

---

## Dataset Description

**Dataset:** Hotel Booking Demand Dataset

The dataset contains reservation-level booking records for two hotel types:

- Resort Hotel
- City Hotel

It includes information on booking lead times, arrival dates, customer demographics, room prices (ADR), cancellations, booking channels, and reservation outcomes.

This project uses the dataset to reconstruct occupancy trends, estimate revenue performance, and build pricing recommendations for hotel revenue managers.

> **Note:** Since the dataset does not contain the hotel's actual room inventory, occupancy will be reconstructed using a synthetic capacity estimation methodology. All pricing recommendations and revenue uplift calculations are presented as simulation-based decision support rather than historical outcomes.

In [1]:
# ==========================================================
# 1. Import Libraries
# ==========================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==========================================================
# 2. Load Dataset
# ==========================================================

df = pd.read_csv("../data/raw/hotel_bookings.csv")

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [3]:
# ==========================================================
# 3. Basic Dataset Information
# ==========================================================

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

df.head()

Rows    : 119,390
Columns : 32


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


### Initial Observations

- The dataset contains **119,390 reservation records** across **32 attributes**.
- Each row represents a single hotel reservation.
- The dataset includes booking details, customer information, pricing (ADR), cancellations, room allocation, and reservation outcomes.
- The dataset size is appropriate for performing exploratory analysis, SQL-based business analytics, and interactive dashboard development.

In [4]:
# ==========================================================
# 4. Dataset Structure
# ==========================================================

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

### Dataset Structure Summary

#### Data Types

The dataset contains a mix of numerical and categorical attributes describing hotel reservations.

- Numerical variables include booking lead time, guest counts, stay duration, ADR, parking requests, booking changes, and previous reservation history.
- Categorical variables describe hotel type, meal plan, customer segment, distribution channel, room types, deposit type, and reservation status.

#### Missing Values

The majority of attributes are complete. Missing values are observed in:

- `children`
- `country`
- `agent`
- `company`

These missing values will be investigated during the data cleaning phase to determine whether they represent missing information or valid business scenarios.

#### Initial Assessment

Overall, the dataset has a well-defined structure with minimal missing data and appears suitable for analytical modelling after data cleaning and feature engineering.

## Key Findings So Far

- Dataset contains 119,390 reservation records across 32 attributes.
- The dataset is reservation-level rather than occupancy-level.
- Most variables have complete data with only a few missing values.
- Reservation status date will require datetime conversion.
- Daily occupancy and room inventory will need to be reconstructed through feature engineering.

In [5]:
# ==========================================================
# 5. Business Column Audit
# ==========================================================

pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.values
})

,Column,Data Type
0,hotel,object
1,is_canceled,int64
2,lead_time,int64
3,arrival_date_year,int64
4,arrival_date_month,object
5,arrival_date_week_number,int64
6,arrival_date_day_of_month,int64
7,stays_in_weekend_nights,int64
8,stays_in_week_nights,int64
9,adults,int64


## Analytical Workflow

```text
Raw Reservation Data
        │
        ▼
Data Cleaning
        │
        ▼
Feature Engineering
        │
        ▼
Demand & Revenue Analysis
        │
        ▼
Occupancy Reconstruction
        │
        ▼
Dynamic Pricing Simulation
        │
        ▼
Executive Dashboards
```

In [6]:
# ==========================================================
# 6. Data Quality Assessment
# ==========================================================

# Missing Values
missing_values = (
    df.isnull()
      .sum()
      .to_frame(name="Missing Values")
)

missing_values["Missing Percentage"] = (
    missing_values["Missing Values"] / len(df) * 100
).round(2)

missing_values = (
    missing_values[missing_values["Missing Values"] > 0]
    .sort_values(by="Missing Values", ascending=False)
)

missing_values

,Missing Values,Missing Percentage
company,112593,94.31
agent,16340,13.69
country,488,0.41
children,4,0.00


### Missing Values Assessment

Four attributes contain missing values:

| Feature | Missing % | Initial Assessment |
|----------|----------:|-------------------|
| company | 94.31% | Most bookings were not made through corporate accounts. This likely represents a valid business scenario rather than poor data quality. |
| agent | 13.69% | May indicate direct bookings or reservations without an intermediary. |
| country | 0.41% | Very small proportion of missing values; can be handled during data cleaning. |
| children | <0.01% | Negligible missing values with minimal impact on analysis. |

Overall, the dataset has excellent completeness for the variables most relevant to revenue management and pricing analysis.

In [7]:
# Duplicate Records

duplicate_count = df.duplicated().sum()

print(f"Duplicate Records : {duplicate_count:,}")

Duplicate Records : 31,994


### Duplicate Records Assessment

A preliminary duplicate check identified **31,994 identical rows**.

However, the dataset does not include a unique booking identifier. Since multiple customers can legitimately make reservations with identical booking characteristics, these records cannot be classified as true duplicates without further investigation.

Duplicate handling will therefore be postponed until the data cleaning phase, where additional business validation will be performed before removing any records.

In [9]:
# ==========================================================
# Investigating Duplicate Reservations
# ==========================================================

duplicates = df[df.duplicated(keep=False)]

print(f"Duplicate Rows (including both occurrences): {duplicates.shape[0]:,}")

duplicates.head(10)

Duplicate Rows (including both occurrences): 40,165


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
5,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
21,Resort Hotel,0,72,2015,July,27,1,2,4,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,A,A,1,No Deposit,250.0,NaN,0,Transient,84.67,0,1,Check-Out,2015-07-07
22,Resort Hotel,0,72,2015,July,27,1,2,4,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,A,A,1,No Deposit,250.0,NaN,0,Transient,84.67,0,1,Check-Out,2015-07-07
39,Resort Hotel,0,70,2015,July,27,2,2,3,2,0.0,0,HB,ROU,Direct,Direct,0,0,0,E,E,0,No Deposit,250.0,NaN,0,Transient,137.00,0,1,Check-Out,2015-07-07
43,Resort Hotel,0,70,2015,July,27,2,2,3,2,0.0,0,HB,ROU,Direct,Direct,0,0,0,E,E,0,No Deposit,250.0,NaN,0,Transient,137.00,0,1,Check-Out,2015-07-07
132,Resort Hotel,1,5,2015,July,28,5,1,0,2,0.0,0,BB,PRT,Online TA,TA/TO,0,0,0,D,D,0,No Deposit,240.0,NaN,0,Transient,97.00,0,0,Canceled,2015-07-01
138,Resort Hotel,1,5,2015,July,28,5,1,0,2,0.0,0,BB,PRT,Online TA,TA/TO,0,0,0,D,D,0,No Deposit,240.0,NaN,0,Transient,97.00,0,0,Canceled,2015-07-01
198,Resort Hotel,0,0,2015,July,28,7,0,1,1,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,109.80,0,3,Check-Out,2015-07-08
200,Resort Hotel,0,0,2015,July,28,7,0,1,1,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,109.80,0,3,Check-Out,2015-07-08


In [10]:
duplicates.describe(include="all")

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
count,40165,40165.000000,40165.000000,40165.000000,40165,40165.000000,40165.000000,40165.000000,40165.000000,40165.000000,40165.000000,40165.000000,40165,40119,40165,40165,40165.000000,40165.000000,40165.000000,40165,40165,40165.000000,40165,34945.000000,2055.000000,40165.000000,40165,40165.000000,40165.000000,40165.000000,40165,40165
unique,2,NaN,NaN,NaN,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,98,7,4,NaN,NaN,NaN,9,11,NaN,3,NaN,NaN,NaN,4,NaN,NaN,NaN,3,844
top,City Hotel,NaN,NaN,NaN,October,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BB,PRT,Groups,TA/TO,NaN,NaN,NaN,A,A,NaN,No Deposit,NaN,NaN,NaN,Transient,NaN,NaN,NaN,Canceled,2015-10-21
freq,31748,NaN,NaN,NaN,5165,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30465,24327,16713,35568,NaN,NaN,NaN,36023,33477,NaN,25773,NaN,NaN,NaN,22209,NaN,NaN,NaN,23181,1376
mean,NaN,0.583867,160.081912,2016.032167,NaN,28.022183,15.796116,0.747019,2.200025,1.799104,0.013295,0.000149,NaN,NaN,NaN,NaN,0.012000,0.207643,0.010059,NaN,NaN,0.101158,NaN,69.155444,209.745012,5.929167,NaN,90.167970,0.004656,0.286667,NaN,NaN
std,NaN,0.492922,126.463447,0.741490,NaN,13.449116,8.657576,0.882845,1.449675,0.430568,0.138002,0.012222,NaN,NaN,NaN,NaN,0.108889,1.378953,0.384438,NaN,NaN,0.382101,NaN,103.453016,133.725002,28.016995,NaN,33.916629,0.068075,0.607798,NaN,NaN
min,NaN,0.000000,0.000000,2015.000000,NaN,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.000000,NaN,1.000000,9.000000,0.000000,NaN,0.000000,0.000000,0.000000,NaN,NaN
25%,NaN,0.000000,56.000000,2015.000000,NaN,17.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.000000,NaN,6.000000,67.000000,0.000000,NaN,65.000000,0.000000,0.000000,NaN,NaN
50%,NaN,1.000000,133.000000,2016.000000,NaN,28.000000,16.000000,0.000000,2.000000,2.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.000000,NaN,14.000000,223.000000,0.000000,NaN,87.000000,0.000000,0.000000,NaN,NaN
75%,NaN,1.000000,251.000000,2017.000000,NaN,40.000000,23.000000,1.000000,3.000000,2.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.000000,NaN,86.000000,308.000000,0.000000,NaN,110.000000,0.000000,0.000000,NaN,NaN


### Duplicate Record Decision

Although the dataset contains **31,994 duplicate records** based on all available attributes, no unique reservation identifier is provided.

As a result, identical rows may represent independent reservations made by different guests rather than true duplicate entries.

To avoid removing potentially valid bookings and introducing bias into occupancy and revenue calculations, these records will be retained throughout the analysis.

This decision preserves the integrity of the reservation data and aligns with the project's objective of modelling real booking demand.

In [11]:
# ==========================================================
# 7. Dataset Snapshot
# ==========================================================

print("Numerical Columns:", df.select_dtypes(include=np.number).shape[1])
print("Categorical Columns:", df.select_dtypes(exclude=np.number).shape[1])

print("\nNumerical Features")
display(df.select_dtypes(include=np.number).columns.to_frame(name="Column"))

print("\nCategorical Features")
display(df.select_dtypes(exclude=np.number).columns.to_frame(name="Column"))

Numerical Columns: 20
Categorical Columns: 12

Numerical Features


,Column
is_canceled,is_canceled
lead_time,lead_time
arrival_date_year,arrival_date_year
arrival_date_week_number,arrival_date_week_number
arrival_date_day_of_month,arrival_date_day_of_month
stays_in_weekend_nights,stays_in_weekend_nights
stays_in_week_nights,stays_in_week_nights
adults,adults
children,children
babies,babies



Categorical Features


,Column
hotel,hotel
arrival_date_month,arrival_date_month
meal,meal
country,country
market_segment,market_segment
distribution_channel,distribution_channel
reserved_room_type,reserved_room_type
assigned_room_type,assigned_room_type
deposit_type,deposit_type
customer_type,customer_type


### Feature Summary

The dataset consists of **20 numerical** and **12 categorical** features.

- Numerical features primarily describe booking behaviour, guest counts, historical booking activity, pricing, and operational metrics.
- Categorical features describe hotel characteristics, customer segments, booking channels, room allocation, and reservation outcomes.

This balanced combination of numerical and categorical variables provides a strong foundation for exploratory analysis, feature engineering, pricing simulation, and business intelligence dashboard development.

# Key Findings

The initial data audit indicates that the dataset is well suited for revenue management analytics.

### Major Findings

- The dataset contains **119,390 hotel reservation records** across **32 attributes**.
- Each record represents an individual reservation rather than daily hotel occupancy.
- The dataset contains **20 numerical** and **12 categorical** features covering customer behaviour, pricing, booking channels, and reservation outcomes.
- Missing values are limited to four attributes and are primarily associated with corporate and agent bookings.
- The dataset does not include hotel capacity or occupancy information, requiring these metrics to be reconstructed during feature engineering.
- Duplicate-looking reservations will be retained because no unique reservation identifier exists, making it impossible to distinguish true duplicate records from legitimate identical bookings.

### Next Steps

The next notebook focuses on preparing the dataset for analysis by:

- Handling missing values
- Validating duplicate records
- Converting date fields
- Engineering new analytical features
- Preparing the dataset for exploratory analysis and SQL-based business analytics